In [90]:
import numpy as np
import scipy.special
import sklearn.datasets

In [91]:
def load_iris():
    """
    Load and preprocess the Iris dataset from sklearn.datasets.
    Returns:
        tuple: A tuple containing two elements.
            - D (numpy.ndarray): Transposed features of the Iris dataset.
            - L (numpy.ndarray): Target labels for the Iris dataset.
    """
    # Import the Iris dataset from sklearn.datasets
    data_dict = sklearn.datasets.load_iris()

    # Extract the feature data and transpose it
    D = data_dict['data'].T

    # Extract the target labels
    L = data_dict['target']

    # Return the transposed features and target labels as a tuple
    return D, L

In [92]:
def split_db_2to1(D, L, seed=0):
    """
    Split the dataset into two parts in a 2:1 ratio.
    Parameters:
        D (numpy.ndarray): Dataset with shape (features, samples)
        L (numpy.ndarray): Target labels with shape (samples,)
        seed (int, optional): Random seed for reproducibility. Defaults to 0.

    Returns:
        tuple: A tuple containing four elements.
            - DTR: Training data with shape (features, samples).
            - LTR: Training labels.
            - DTE: Test data with shape (features, samples).
            - LTE: Test labels.
    """
    # Calculate the number of samples for the training set
    nTrain = int(D.shape[1]*2.0/3.0)

    # Set the seed for random number generator
    np.random.seed(seed)

    # Shuffle the indices of the dataset
    idx = np.random.permutation(D.shape[1])

    # Split the shuffled indices into training and testing sets
    idxTrain = idx[:nTrain]
    idxTest = idx[nTrain:]

    # Select the samples for the training and test sets
    DTR = D[:, idxTrain]
    DTE = D[:, idxTest]

    # Select the target labels
    LTR = L[idxTrain]
    LTE = L[idxTest]

    # Return without transposing to maintain features as rows, samples as columns
    return DTR, LTR, DTE, LTE


In [93]:
def vcol(x):
    """
    Convert a 1D numpy array to a column vector.

    Args:
        x (numpy.ndarray): A 1D numpy array that needs to be converted.

    Returns:
        numpy.ndarray: The input array reshaped as a column vector.
    """
    return x.reshape(x.size, 1)

In [94]:
def vrow(x):
    """
    Convert a 1D numpy array to a row vector.

    Parameters:
    x (numpy.ndarray): A 1D numpy array that needs to be converted to a row vector.

    Returns:
    numpy.ndarray: The input array reshaped as a row vector.
    """
    return x.reshape(1, x.size)

In [95]:
# Load the Iris dataset and separate it into data (D) and labels (L)
D, L = load_iris()
print(D.shape)  # Output: (4, 150), indicating there are 4 features with 150 samples
print(L.shape)  # Output: (150,), indicating there are 150 labels corresponding to the samples

# Split the dataset into training and testing sets in a 2:1 ratio
DTR, LTR, DTE, LTE = split_db_2to1(D, L)

(4, 150)
(150,)


In [96]:
def compute_class_parameters(DTR, LTR):
    """
    Compute class parameters (mean and covariance) for each class.
    Parameters:
        DTR (numpy.ndarray): Training data with shape (features, samples)
        LTR (numpy.ndarray): Training labels with shape (samples,)

    Returns:
        tuple: Two dictionaries containing mean vectors and covariance matrices
    """
    # Extract unique classes from the labels
    classes = np.unique(LTR)

    # Dictionaries to store mean (mu) and covariance (sigma) for each class
    mu_dict = {}
    sigma_dict = {}

    # Iterate over each class
    for c in classes:
        # Select data points belonging to class c
        # Use column indexing since samples are columns
        X_c = DTR[:, LTR == c]

        # Calculate the mean of class c along axis 1 (samples)
        mu_c = X_c.mean(1).reshape((-1, 1))

        # Center the data by subtracting the mean
        centered_data = X_c - mu_c

        # Calculate the covariance matrix for class c
        sigma_c = np.dot(centered_data, centered_data.T) / X_c.shape[1]

        # Store the calculated mean and covariance in dictionaries
        mu_dict[c] = mu_c
        sigma_dict[c] = sigma_c

    # Return the dictionaries containing means and covariances for each class
    return mu_dict, sigma_dict

In [97]:
# Compute the class parameters for each class in the training data
mu_dict, sigma_dict = compute_class_parameters(DTR, LTR)

# Iterate over each class label (0, 1, 2) and print the mean (mu) and covariance (Sigma) matrices
for c in [0, 1, 2]:
    # Print the mean vector for class 'c', reshaped into a column vector
    print(f"mu_{c} =", mu_dict[c].reshape(-1))

    # Print the covariance matrix for class 'c' with proper formatting
    print(f"Sigma_{c} =\n", sigma_dict[c])

mu_0 = [4.96129032 3.42903226 1.46451613 0.2483871 ]
Sigma_0 =
 [[0.13140479 0.11370447 0.02862643 0.01187305]
 [0.11370447 0.16270552 0.01844953 0.01117586]
 [0.02862643 0.01844953 0.03583767 0.00526535]
 [0.01187305 0.01117586 0.00526535 0.0108845 ]]
mu_1 = [5.91212121 2.78484848 4.27272727 1.33939394]
Sigma_1 =
 [[0.26470156 0.09169881 0.18366391 0.05134068]
 [0.09169881 0.10613407 0.08898072 0.04211203]
 [0.18366391 0.08898072 0.21955923 0.06289256]
 [0.05134068 0.04211203 0.06289256 0.03208448]]
mu_2 = [6.45555556 2.92777778 5.41944444 1.98888889]
Sigma_2 =
 [[0.30080247 0.08262346 0.18614198 0.04311728]
 [0.08262346 0.08533951 0.06279321 0.05114198]
 [0.18614198 0.06279321 0.18434414 0.04188272]
 [0.04311728 0.05114198 0.04188272 0.0804321 ]]


In [98]:
def logpdf_GAU_ND(X, mu, Sigma):
    """
    Compute log-density of multivariate Gaussian
    """

    # Number of features (dimensions)
    M = X.shape[0]

    # Centered data: subtract the mean vector from each data point
    XC = X - mu

    # Constant term for the log-density formula
    const = -0.5 * M * np.log(2*np.pi)

    # Compute the logarithm of the determinant and the sign (should be positive for a valid covariance matrix)
    sign, logdet = np.linalg.slogdet(Sigma)

    # Inverse of the covariance matrix
    inv_sigma = np.linalg.inv(Sigma)

    # Initialize an array to store the log-density for each data point
    log_densities = np.zeros(X.shape[1])

    # Loop through each data point to compute its log-density
    for i in range(X.shape[1]):
        xc = XC[:, i:i+1]  # Extract the centered data point (column vector)
        log_densities[i] = const - 0.5 * logdet - 0.5 * np.dot(np.dot(xc.T, inv_sigma), xc)  # Compute the log-density

    return log_densities

In [99]:
def compute_log_densities(X, mu_dict, sigma_dict):
    """
    Compute log-densities for all classes
    """
    # Initialize an array to store the log densities for each class and data point
    S = np.zeros((len(mu_dict), X.shape[1]))

    # Iterate over each class in the dictionary of means (mu_dict)
    for c in mu_dict.keys():
        # Compute the log density for each data point with respect to the current class's mean and covariance matrix
        S[c, :] = logpdf_GAU_ND(X, mu_dict[c], sigma_dict[c])

    # Return the array of log densities
    return S

In [100]:
# Compute log-densities for test data
S = compute_log_densities(DTE, mu_dict, sigma_dict)
print(S)

[[ 1.55967572e+00  1.14017174e+00 -1.40657517e+02 -4.17161807e+02
  -4.79793093e+02  2.41007437e+00  2.10494245e+00  1.67400801e+00
  -1.70961371e+02 -1.66665979e+02  7.06271117e-01 -9.73464551e-01
  -1.51986858e+02 -2.45521190e-01 -5.63754861e+02 -7.67214078e+01
  -4.20761218e+02 -1.65272890e+02 -7.85138635e-01 -2.42586918e+02
  -4.06216307e+00 -4.88360403e+02  2.17363495e+00  1.81394319e+00
  -3.95752259e+02  2.37180771e+00 -4.04665187e+02 -1.38807386e+02
  -1.18431556e+02 -8.29839492e+01 -3.56324460e+02 -3.48262925e+02
  -2.27427204e+02 -2.46143837e+02  8.75894828e-01 -9.77609722e+01
  -4.03297575e+02 -2.91264978e+02  2.38446728e+00 -1.76880387e+02
  -1.37384611e+02 -2.41764742e+02 -1.73726267e+02 -1.47937484e+00
   1.59429843e+00  1.15000214e+00 -3.23352247e+02 -1.20234441e+02
  -4.86995891e+02  2.22922516e+00]
 [-5.23573716e+01 -7.42164838e+01 -1.35485877e-01 -1.97569706e+01
  -1.78065522e+01 -7.33764524e+01 -5.53810038e+01 -7.00532948e+01
   4.27477373e-01  2.33201576e-01 -6.4320

C:\Users\marca\AppData\Local\Temp\ipykernel_34376\1296313468.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  log_densities[i] = const - 0.5 * logdet - 0.5 * np.dot(np.dot(xc.T, inv_sigma), xc)  # Compute the log-density


In [101]:
# Convert log-densities to densities
S_exp = np.exp(S)

# Assuming uniform priors P(c) = 1/3
prior = np.ones(3) / 3

# Compute joint densities
SJoint = S_exp * vcol(prior)

# Compute marginal densities
SMarginal = vrow(SJoint.sum(0))

# Compute posterior probabilities
SPost = SJoint / SMarginal

# Make predictions
predictions = np.argmax(SPost, axis=0)

# Compute error rate
error_rate = (predictions != LTE).mean()
print(f"MVG Error Rate (direct approach): {error_rate*100:.1f}%")

MVG Error Rate (direct approach): 4.0%


In [102]:
# Compute log-prior
logPrior = np.log(prior)

# Compute joint log-densities
logSJoint = S + vcol(logPrior)

# Compute marginal log-densities using log-sum-exp trick
logSMarginal = vrow(scipy.special.logsumexp(logSJoint, axis=0))

# Compute log-posteriors
logSPost = logSJoint - logSMarginal

# Convert to posteriors
SPost_log = np.exp(logSPost)

# Make predictions
predictions_log = np.argmax(SPost_log, axis=0)

# Compute error rate
error_rate_log = (predictions_log != LTE).mean()
print(f"MVG Error Rate (log-domain approach): {error_rate_log*100:.1f}%")

MVG Error Rate (log-domain approach): 4.0%


In [103]:
def naive_bayes_params(mu_dict, sigma_dict):
    """
    Create Naive Bayes parameters by zeroing off-diagonal elements
    """

    # Copy the mean dictionary to avoid modifying the original data
    mu_NB = mu_dict.copy()

    # Initialize an empty dictionary to store the modified covariance matrices
    sigma_NB = {}

    # Iterate over each class in the mean dictionary
    for c in mu_dict.keys():
        # Zero out off-diagonal elements by element-wise multiplying with identity matrix
        sigma_NB[c] = sigma_dict[c] * np.eye(sigma_dict[c].shape[0])

    # Return the modified mean and covariance dictionaries
    return mu_NB, sigma_NB

In [104]:
# Compute Naive Bayes parameters
mu_NB, sigma_NB = naive_bayes_params(mu_dict, sigma_dict)

# Compute log-densities for test data
S_NB = compute_log_densities(DTE, mu_NB, sigma_NB)

# Compute posteriors
logSJoint_NB = S_NB + vcol(logPrior)
logSMarginal_NB = vrow(scipy.special.logsumexp(logSJoint_NB, axis=0))
logSPost_NB = logSJoint_NB - logSMarginal_NB
SPost_NB = np.exp(logSPost_NB)

# Make predictions and compute error rate
predictions_NB = np.argmax(SPost_NB, axis=0)
error_rate_NB = (predictions_NB != LTE).mean()
print(f"Naive Bayes Error Rate: {error_rate_NB*100:.1f}%")

Naive Bayes Error Rate: 4.0%


C:\Users\marca\AppData\Local\Temp\ipykernel_34376\1296313468.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  log_densities[i] = const - 0.5 * logdet - 0.5 * np.dot(np.dot(xc.T, inv_sigma), xc)  # Compute the log-density


In [105]:

def tied_cov_params(mu_dict, sigma_dict, DTR, LTR):
    """
    Compute tied covariance matrix
    """

    # Copy the means to ensure we don't modify the original dictionary
    mu_tied = mu_dict.copy()

    # Initialize the tied covariance matrix with zeros,
    # assuming the same shape as the covariance matrices in sigma_dict
    sigma_tied = np.zeros_like(sigma_dict[0])

    # Get the total number of samples across all classes
    N = DTR.shape[1]  # Total number of samples

    # Iterate over each class
    for c in mu_dict.keys():
        # Count the number of samples belonging to class c
        nc = np.sum(LTR == c)

        # Update the tied covariance matrix by adding the weighted
        # covariance of class c to the cumulative sum
        sigma_tied += (nc / N) * sigma_dict[c]

    # Create a dictionary where each class has the same tied covariance matrix
    sigma_tied_dict = {c: sigma_tied for c in mu_dict.keys()}

    return mu_tied, sigma_tied_dict

In [106]:
# Compute tied covariance parameters
mu_tied, sigma_tied_dict = tied_cov_params(mu_dict, sigma_dict, DTR, LTR)

# Display tied covariance matrix
print("Tied Covariance Matrix:")
print(sigma_tied_dict[0])

# Compute log-densities
S_tied = compute_log_densities(DTE, mu_tied, sigma_tied_dict)

# Compute posteriors
logSJoint_tied = S_tied + vcol(logPrior)
logSMarginal_tied = vrow(scipy.special.logsumexp(logSJoint_tied, axis=0))
logSPost_tied = logSJoint_tied - logSMarginal_tied
SPost_tied = np.exp(logSPost_tied)

# Make predictions and compute error rate
predictions_tied = np.argmax(SPost_tied, axis=0)
error_rate_tied = (predictions_tied != LTE).mean()
print(f"Tied Covariance Error Rate: {error_rate_tied*100:.1f}%")

Tied Covariance Matrix:
[[0.23637589 0.09525344 0.1364944  0.03614529]
 [0.09525344 0.11618517 0.05768855 0.0357726 ]
 [0.1364944  0.05768855 0.14992811 0.03746458]
 [0.03614529 0.0357726  0.03746458 0.04291763]]
Tied Covariance Error Rate: 2.0%


C:\Users\marca\AppData\Local\Temp\ipykernel_34376\1296313468.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  log_densities[i] = const - 0.5 * logdet - 0.5 * np.dot(np.dot(xc.T, inv_sigma), xc)  # Compute the log-density


In [107]:
def extract_binary_dataset(D, L, class1=1, class2=2):
    """
    Extract binary dataset with only two classes
    """

    # Create a mask to identify samples that belong to either class1 or class2
    mask = np.logical_or(L == class1, L == class2)

    # Extract the data samples that match the mask
    D_binary = D[:, mask]

    # Extract the labels for the binary dataset
    L_binary = L[mask]

    # Map the labels to binary values: 0 for class1 and 1 for class2
    L_binary = np.where(L_binary == class2, 1, 0)

    return D_binary, L_binary

In [109]:
# Extract binary dataset
D_binary, L_binary = extract_binary_dataset(D, L, 1, 2)

# Split dataset
DTR_binary, LTR_binary, DTE_binary, LTE_binary = split_db_2to1(D_binary, L_binary)

# Compute MVG parameters
mu_binary, sigma_binary = compute_class_parameters(DTR_binary, LTR_binary)

# Compute log-densities
S_binary = compute_log_densities(DTE_binary, mu_binary, sigma_binary)

# Compute log-likelihood ratio: log(p(x|c=1) / p(x|c=0))
LLR = S_binary[1, :] - S_binary[0, :]

# Make predictions (threshold = 0 for uniform prior)
predictions_binary = (LLR >= 0).astype(int)

# Compute error rate
error_rate_binary = (predictions_binary != LTE_binary).mean()
print(f"Binary MVG Error Rate: {error_rate_binary*100:.1f}%")

Binary MVG Error Rate: 8.8%


C:\Users\marca\AppData\Local\Temp\ipykernel_34376\1296313468.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  log_densities[i] = const - 0.5 * logdet - 0.5 * np.dot(np.dot(xc.T, inv_sigma), xc)  # Compute the log-density


In [110]:
# Compute tied covariance parameters
mu_binary_tied, sigma_binary_tied = tied_cov_params(mu_binary, sigma_binary, DTR_binary, LTR_binary)

# Compute log-densities
S_binary_tied = compute_log_densities(DTE_binary, mu_binary_tied, sigma_binary_tied)

# Compute log-likelihood ratio
LLR_tied = S_binary_tied[1, :] - S_binary_tied[0, :]

# Make predictions (threshold = 0)
predictions_binary_tied = (LLR_tied >= 0).astype(int)

# Compute error rate
error_rate_binary_tied = (predictions_binary_tied != LTE_binary).mean()
print(f"Binary Tied Covariance Error Rate: {error_rate_binary_tied*100:.1f}%")

Binary Tied Covariance Error Rate: 5.9%


C:\Users\marca\AppData\Local\Temp\ipykernel_34376\1296313468.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  log_densities[i] = const - 0.5 * logdet - 0.5 * np.dot(np.dot(xc.T, inv_sigma), xc)  # Compute the log-density
